# FTT (FT-Transformer) fold0 baseline — exp_069

NN attention 축(ftt.md, #031 NN 신축). features=realmlp_fe_v2(피처 통제, 메커니즘 분기 테스트), FTT_D default 무튜닝. **게이트: fold0 corr<0.97(분기 1차) + 개별~0.95.** skorch 의존.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 감지 → 조건부 torch + 프로젝트 deps(+skorch FTT 의존)
import sys, subprocess
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch 재설치')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등 → 기본 torch 유지')
pip('pytabkit','skorch','hydra-core','python-dotenv')

In [ ]:
# 3) torch 검증 + train_ftt import
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x=torch.randn(256,256,device='cuda'); _=(_x@_x).sum().item(); print('CUDA OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_ftt import run
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg + run — FTT baseline(default), realmlp_fe_v2, fold0
from omegaconf import OmegaConf
import time
CONF = Path(SRC_ROOT) / 'conf'
cfg = OmegaConf.create({
    'exp_id': 'exp_069_ftt_baseline_fold0',
    'notes': 'FTT_D baseline fold0: realmlp_fe_v2, default 무튜닝, NN attention 축 분기 테스트',
    'use_wandb': False, 'max_folds': 1,
    'model': OmegaConf.load(CONF / 'model' / 'ftt.yaml'),
    'features': OmegaConf.load(CONF / 'features' / 'realmlp_fe_v2.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
t0=time.time(); result=run(cfg); print(result, f'{time.time()-t0:.0f}s')
print('fold0 AUC =', result.get('fold_scores',[None])[0])